# Bottleneck Score — Part 2: LLM-Upgraded Supply Chain Graph Analysis

**Author:** Leo (Seunguk Lee)  
**Series:** [Part 1](https://medium.com/@hugesisulee/why-cheap-stocks-fail-and-how-supply-chain-graph-analysis-fixes-it-627c86d38108) → Part 2 (this notebook)  
**Reference:** *"The contribution of LLMs to relation extraction in the economic field"* (2025)

---

## Overview

Part 1 used keyword proximity matching to extract supply chain relationships from SEC 10-K filings.  
This notebook upgrades that pipeline with two LLM-based classifiers and compares their investment outcomes.

### Experiment Structure

| Approach | Method | Prompt Philosophy |
|---|---|---|
| **V1: Keyword** | Regex proximity match | Does the company name appear near a supply-chain keyword? |
| **V2: Structural LLM** | Llama 3.3 via Groq API | What type of relationship does this sentence describe? |
| **V3: Predictive LLM** | Llama 3.1 via Ollama (local) | If Company A grows, will demand for Company B inevitably increase? |

### Key Finding
The **prompt framing** — not just model capability — meaningfully changes which companies are identified as network bottlenecks, and this difference propagates into portfolio performance.

### Known Limitations
- V2 uses Llama 3.3 (Groq), V3 uses Llama 3.1 (Ollama local). Model versions differ, so performance gaps cannot be attributed to prompt alone.
- Backtest windows are limited (1-year and 3-year bull market). Results should not be interpreted as robust alpha.
- 150-character context window may be insufficient for nuanced relationship inference.

---

## Setup

```bash
pip install yfinance requests pandas networkx matplotlib beautifulsoup4 groq ollama
```

> **Security Note:** Never hard-code API keys in notebooks you plan to share.  
> Store them as environment variables and load via `os.environ.get("GROQ_API_KEY")`.


## 1. Shared Configuration & Utilities

Ticker universe, SEC headers, CIK mappings, and the 10-K URL fetcher shared across all three experiments.

In [19]:
import os
import yfinance as yf
import requests
import json
import time
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import re
from bs4 import BeautifulSoup

# ── Target universe ────────────────────────────────────────────────────────────
IT_TICKERS = [
    'MSFT', 'AAPL', 'NVDA', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'CSCO', 'AMD', 'QCOM',
    'TXN', 'INTU', 'IBM', 'AMAT', 'MU', 'NOW', 'LRCX', 'ADI', 'PANW', 'KLAC',
    'SNPS', 'CDNS', 'MSI', 'APH', 'CDW', 'TEL', 'FTNT', 'ANET', 'KEYS', 'GLW',
    'TER', 'STX', 'NTAP', 'FSLR', 'TYL', 'AKAM', 'GEN', 'JNPR', 'QRVO', 'SWKS',
    'WDC', 'ENPH', 'TRMB', 'ZBRA', 'PTC', 'VRT', 'ETN', 'HPE'
]
BIG_TECH = ['MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA']  # CapEx demand anchor

# ── SEC EDGAR headers ──────────────────────────────────────────────────────────
# Per SEC fair-access policy, provide a valid contact email in User-Agent.
HEADERS = {
    'User-Agent': 'Academic ResearchProject your email',  # ⚠️ Replace with your email
    'Accept-Encoding': 'gzip, deflate'
}

# ── CIK & company name mappings ───────────────────────────────────────────────
TICKER_TO_CIK = {
    'MSFT': '0000789019', 'AAPL': '0000320193', 'NVDA': '0001045810', 'AVGO': '0001730168',
    'ORCL': '0001341439', 'CRM': '0001108524', 'ADBE': '0000796343', 'CSCO': '0000858877',
    'AMD': '0000002488', 'QCOM': '0000804328', 'TXN': '0000097476', 'INTU': '0000896878',
    'IBM': '0000051143', 'AMAT': '0000006951', 'MU': '0000723125', 'NOW': '0001373715',
    'LRCX': '0000707549', 'ADI': '0000006281', 'PANW': '0001327568', 'KLAC': '0000319201',
    'SNPS': '0000883241', 'CDNS': '0000813672', 'MSI': '0000068505', 'APH': '0000820313',
    'CDW': '0001402064', 'TEL': '0001385157', 'FTNT': '0001262039', 'ANET': '0001591862',
    'KEYS': '0001601046', 'GLW': '0000024741', 'TER': '0000097210', 'STX': '0001137789',
    'NTAP': '0001002047', 'FSLR': '0001274494', 'TYL': '0000860731', 'AKAM': '0001086222',
    'GEN': '0000849318', 'JNPR': '0001043604', 'QRVO': '0001602658', 'SWKS': '0000004127',
    'WDC': '0001060394', 'ENPH': '0001463101', 'TRMB': '0000864749', 'ZBRA': '0000877212',
    'PTC': '0000857002', 'VRT': '0001774759', 'ETN': '0001551182', 'HPE': '0001645590'
}

TICKER_TO_NAME = {
    'MSFT': 'Microsoft', 'AAPL': 'Apple', 'NVDA': 'NVIDIA', 'AVGO': 'Broadcom',
    'ORCL': 'Oracle', 'CRM': 'Salesforce', 'ADBE': 'Adobe', 'CSCO': 'Cisco Systems',
    'AMD': 'Advanced Micro Devices', 'QCOM': 'QUALCOMM', 'TXN': 'Texas Instruments',
    'INTU': 'Intuit', 'IBM': 'International Business Machines', 'AMAT': 'Applied Materials',
    'MU': 'Micron Technology', 'NOW': 'ServiceNow', 'LRCX': 'Lam Research',
    'ADI': 'Analog Devices', 'PANW': 'Palo Alto Networks', 'KLAC': 'KLA',
    'SNPS': 'Synopsys', 'CDNS': 'Cadence Design Systems', 'MSI': 'Motorola Solutions',
    'APH': 'Amphenol', 'CDW': 'CDW', 'TEL': 'TE Connectivity', 'FTNT': 'Fortinet',
    'ANET': 'Arista Networks', 'KEYS': 'Keysight Technologies', 'GLW': 'Corning',
    'TER': 'Teradyne', 'STX': 'Seagate Technology', 'NTAP': 'NetApp',
    'FSLR': 'First Solar', 'TYL': 'Tyler Technologies', 'AKAM': 'Akamai Technologies',
    'GEN': 'Gen Digital', 'JNPR': 'Juniper Networks', 'QRVO': 'Qorvo',
    'SWKS': 'Skyworks Solutions', 'WDC': 'Western Digital', 'ENPH': 'Enphase Energy',
    'TRMB': 'Trimble', 'ZBRA': 'Zebra Technologies', 'PTC': 'PTC',
    'VRT': 'Vertiv Holdings', 'ETN': 'Eaton', 'HPE': 'Hewlett Packard Enterprise'
}


def get_10k_url(cik: str, date_range: tuple) -> str | None:
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    start, end = date_range
    try:
        res = requests.get(url, headers=HEADERS)
        if res.status_code != 200:
            return None
        submissions = res.json()['filings']['recent']
        for i, form in enumerate(submissions['form']):
            filing_date = submissions['filingDate'][i]
            if form == '10-K' and start <= filing_date <= end:
                acc = submissions['accessionNumber'][i].replace('-', '')
                doc = submissions['primaryDocument'][i]
                return f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}/{acc}/{doc}"
    except Exception:
        pass
    return None


def fetch_big_tech_capex(tickers: list, max_year: int = 2024) -> pd.Series:
    capex_list = []
    for t in tickers:
        try:
            cf = yf.Ticker(t).cash_flow.T
            if cf is None or cf.empty:
                print(f"  ⚠️ [{t}] Cash flow data is empty (yfinance blocking/issue).")
                continue
                
            if 'Capital Expenditure' not in cf.columns:
                print(f"  ⚠️ [{t}] 'Capital Expenditure' column not found.")
                continue
                
            capex = cf['Capital Expenditure'].abs()
            capex.index = pd.to_datetime(capex.index).year
            
            # ✅ 에러 수정: 필터링을 먼저 한 후, 필터링된 결과의 index로 그룹화
            capex = capex[capex.index <= max_year]
            capex = capex.groupby(capex.index).last()
            
            capex_list.append(capex)
            
        except Exception as e:
            print(f"  ❌ [{t}] Error fetching CapEx: {e}")
            continue
            
    if not capex_list:
        print("  🚨 API failed for all Big Tech. Using robust fallback data for CapEx anchor.")
        fallback = pd.Series({
            2020: 100e9, 
            2021: 130e9, 
            2022: 150e9, 
            2023: 160e9, 
            2024: 180e9
        })
        return fallback[fallback.index <= max_year]
        
    return pd.concat(capex_list, axis=1).sum(axis=1)


def compute_bottleneck_score(
    revenue_series: pd.Series,  # ✅ API 호출 방지를 위해 캐싱된 매출 데이터 사용
    in_degree_weighted: float,
    max_in: float,
    pe: float,
    total_capex: pd.Series,
    max_year: int = 2024
) -> float | None:
    if revenue_series is None or revenue_series.empty:
        return None
    try:
        rev = revenue_series.copy()
        rev.index = pd.to_datetime(rev.index).year
        rev = rev[rev.index <= max_year].groupby(rev.index).last()
        
        combined = pd.DataFrame({'Capex': total_capex, 'Rev': rev}).dropna()
        if len(combined) < 2:
            return None
            
        corr = combined.corr().iloc[0, 1]
        norm_in = in_degree_weighted / max_in
        score = ((norm_in + 0.1) * (corr + 1)) / (max(pe, 5) / 10)
        return round(score, 4)
    except Exception:
        return None


def plot_network(G: nx.DiGraph, score_df: pd.DataFrame, profiles: dict, title: str):
    plt.figure(figsize=(16, 10))
    pos = nx.spring_layout(G, k=0.5, seed=42)
    node_sizes = [
        score_df.loc[n, 'Bottleneck_Score'] * 2000 + 300
        if n in score_df.index else 300
        for n in G.nodes()
    ]
    node_colors = [profiles[n]['pe'] if n in profiles else 30 for n in G.nodes()]
    nodes = nx.draw_networkx_nodes(
        G, pos, node_size=node_sizes, node_color=node_colors,
        cmap=plt.cm.coolwarm, edgecolors='black'
    )
    nx.draw_networkx_edges(G, pos, alpha=0.3, arrowstyle='->', arrowsize=12, edge_color='gray')
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.colorbar(nodes, label='Estimated P/E Ratio')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Shared config loaded.")
print(f"   Universe: {len(IT_TICKERS)} IT companies | Big Tech anchor: {BIG_TECH}")

✅ Shared config loaded.
   Universe: 48 IT companies | Big Tech anchor: ['MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA']


## 2. V2: Structural LLM Classifier (Groq — Llama 3.3 70B)

**Prompt philosophy:** *"What type of relationship does this sentence currently describe?"*

The LLM classifies each 10-K mention into: `partner | customer | supplier | competitor | none`,  
and assigns a confidence-weighted edge weight based on relationship type.

**Model:** Llama 3.3 70B via [Groq API](https://console.groq.com) (free tier available)

> **API Key Setup:**  
> Set `GROQ_API_KEY` as an environment variable before running.  
> On Mac/Linux: `export GROQ_API_KEY=gsk_...`  
> On Windows: `set GROQ_API_KEY=gsk_...`


In [ ]:
import ollama
import os

RELATIONSHIP_WEIGHTS = {
    "partner":    1.0,
    "customer":   0.8,
    "supplier":   0.6,
    "competitor": 0.2,
    "none":       0.0
}


def classify_structural(sentence: str, company_a: str, company_b: str) -> dict:
    prompt = f"""You are analyzing SEC 10-K filings to classify business relationships.
Company A (the filing company): {company_a}
Company B (mentioned company): {company_b}
Sentence from the 10-K: \"{sentence}\"

Classify the relationship from Company A's perspective.
Reply with a SINGLE valid JSON object and nothing else. No explanation, no markdown.

{{"relationship_type": "partner|customer|supplier|competitor|none",
  "confidence": 0.0-1.0,
  "direction": "A_depends_on_B|B_depends_on_A|mutual|none"}}

Rules:
- partner: strategic alliance, joint development
- customer: B buys from A (A earns revenue from B)
- supplier: A buys from B (A depends on B for components/services)
- competitor: both compete in same market
- none: incidental mention, no real business relationship"""

    try:
        response = ollama.chat(
            model='llama3.1',
            messages=[{'role': 'user', 'content': prompt}]
            # format: json 제거 — 프롬프트로 직접 강제
        )
        raw = response['message']['content'].strip()

        # JSON 블록이 여러 개 붙어 나올 때 첫 번째만 추출
        first_end = raw.find('}')
        if first_end == -1:
            raise ValueError("No JSON object found")
        raw = raw[:first_end + 1]

        # 혹시 ```json ... ``` 마크다운으로 감싸진 경우 제거
        raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()

        result     = json.loads(raw)
        rel_type   = result.get("relationship_type", "none")
        confidence = float(result.get("confidence", 0.0))
        direction  = result.get("direction", "none")
        weight     = RELATIONSHIP_WEIGHTS.get(rel_type, 0.0) * confidence

        return {
            "relationship_type": rel_type,
            "confidence": confidence,
            "direction": direction,
            "weight": weight
        }
    except Exception as e:
        print(f"  ❌ Ollama error: {e}")
        return {"relationship_type": "none", "confidence": 0.0, "direction": "none", "weight": 0.0}


def run_structural_llm_analysis(
    tickers: list,
    big_tech: list,
    snapshot_start: str = "2025-05-12",
    snapshot_end: str = "2025-05-17",
    filing_range: tuple = ("2024-01-01", "2025-05-15"),
    weight_threshold: float = 0.3
) -> pd.DataFrame:

    total_capex = fetch_big_tech_capex(big_tech)
    if total_capex.empty:
        print("❌ Could not retrieve Big Tech CapEx data.")
        return pd.DataFrame()

    # ── Step 1: Build company profiles ───────────────────────────────────────
    print("Step 1: Building company profiles with historical P/E...")
    profiles = {}
    valid = [t for t in tickers if t in TICKER_TO_CIK]
    for t in valid:
        try:
            stock = yf.Ticker(t)
            hist  = stock.history(start=snapshot_start, end=snapshot_end)
            price = hist['Close'].iloc[-1] if not hist.empty else None

            fin     = stock.financials.T
            revenue = fin['Total Revenue'] if 'Total Revenue' in fin.columns else None

            pe = 30  # fallback
            fin.index = pd.to_datetime(fin.index).year
            if 2024 in fin.index and price:
                row = fin.loc[2024]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                eps = next((row[c] for c in fin.columns if 'EPS' in c or 'Earnings Per Share' in c), None)
                if (eps is None or pd.isna(eps) or eps <= 0) and 'Net Income' in row:
                    shares = row.get('Diluted Average Shares', row.get('Basic Average Shares', 1))
                    eps = row['Net Income'] / shares if shares > 0 else None
                if eps and eps > 0:
                    pe = price / eps

            profiles[t] = {'name': TICKER_TO_NAME[t], 'pe': pe, 'revenue': revenue}
        except Exception:
            continue
    print(f"  → {len(profiles)} companies profiled.")

    # ── Step 2: Extract relationships from 10-K filings ──────────────────────
    print("Step 2: Scraping 10-K filings and classifying relationships...")
    edges, edge_types = [], {}
    for t, prof in profiles.items():
        url = get_10k_url(TICKER_TO_CIK[t], filing_range)
        if not url:
            continue
        try:
            time.sleep(0.2)  # Respect SEC rate limit
            res = requests.get(url, headers=HEADERS)
            if res.status_code != 200:
                continue
            text = re.sub(r'\s+', ' ', BeautifulSoup(res.text, 'html.parser').get_text(separator=' ').lower())

            for target, tprof in profiles.items():
                if target == t:
                    continue

                name_lower = tprof['name'].lower()
                if name_lower not in text:
                    continue

                # First mention only — keeps speed consistent with original
                idx     = text.find(name_lower)
                snippet = text[max(0, idx - 150): idx + 150]
                result  = classify_structural(snippet, prof['name'], tprof['name'])

                weight    = result['weight']
                direction = result['direction']

                if weight > weight_threshold:
                    if direction == 'A_depends_on_B':
                        edges.append((t, target, weight))
                    elif direction == 'B_depends_on_A':
                        edges.append((target, t, weight))
                    elif direction == 'mutual':
                        edges.append((t, target, weight))
                        edges.append((target, t, weight))

                    edge_types[(t, target)] = result['relationship_type']

        except Exception:
            continue
    print(f"  → {len(edges)} edges extracted.")

    if not edges:
        print("❌ No edges found. Check HEADERS email and network access.")
        return pd.DataFrame()

    # ── Step 3: Build graph and compute Bottleneck Scores ────────────────────
    print("Step 3: Building graph and computing Bottleneck Scores...")
    G = nx.DiGraph()
    G.add_weighted_edges_from(edges)
    in_degrees = dict(G.in_degree(weight='weight'))
    max_in     = max(in_degrees.values(), default=1)

    results = []
    for t in G.nodes():
        if t not in profiles:
            continue

        score = compute_bottleneck_score(
            profiles[t]['revenue'],
            in_degrees.get(t, 0),
            max_in,
            profiles[t]['pe'],
            total_capex
        )

        if score is None:
            print(f"  ⚠️ [{t}] Score calculation failed — missing revenue or CapEx overlap")
        else:
            results.append({
                'Ticker':             t,
                'In_Degree_Weighted': round(in_degrees.get(t, 0), 3),
                'PE':                 round(profiles[t]['pe'], 2),
                'Bottleneck_Score':   score
            })

    if not results:
        print("❌ Edges found but no scores computed — check revenue data.")
        return pd.DataFrame()

    df = pd.DataFrame(results).set_index('Ticker')
    plot_network(G, df, profiles, "V2: Structural LLM — Supply Chain Network (Ollama Llama 3.1)")
    return df.sort_values('Bottleneck_Score', ascending=False)


# ── Run ───────────────────────────────────────────────────────────────────────
print("Running V2: Structural LLM (Ollama)...")
v2_results = run_structural_llm_analysis(IT_TICKERS, BIG_TECH)
if not v2_results.empty:
    print("\nTop 20 by Bottleneck Score (V2 — Structural LLM):")
    print(v2_results.head(20))

Running V2: Structural LLM (Ollama)...
Step 1: Building company profiles with historical P/E...


$JNPR: possibly delisted; no timezone found


  → 48 companies profiled.
Step 2: Scraping 10-K filings and classifying relationships...
  ❌ Ollama error: Extra data: line 7 column 1 (char 87)
  ❌ Ollama error: Extra data: line 7 column 1 (char 85)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Extra data: line 7 column 1 (char 79)
  ❌ Ollama error: Extra data: line 7 column 1 (char 79)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Extra data: line 7 column 1 (char 79)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Expecting value: line 1 column 1 (char 0)
  ❌ Ollama error: Extra data: line 7 column 1 (char 83)
  ❌ Ollama error: Expecting value:

KeyboardInterrupt: 

## 3. V3: Predictive LLM Classifier (Ollama — Llama 3.1, local)

**Prompt philosophy:** *"If Company A expands, will demand for Company B inevitably increase?"*

This is a fundamentally different question. Instead of describing a current relationship,  
the model reasons about **structural demand transfer** and **substitutability** — two concepts  
that map more directly to durable competitive moats.

**Edge weight formula:**  
`weight = demand_leverage_score × substitutability_multiplier`

Where substitutability multiplier: `low=1.0`, `medium=0.5`, `high=0.1`

**Setup:** Install and pull the model locally before running.
```bash
# Install Ollama: https://ollama.com
ollama pull llama3.1
```

> **Limitation:** This cell uses Llama 3.1 (local) while V2 used Llama 3.3 (Groq).  
> Performance differences between V2 and V3 therefore reflect both prompt design *and* model version.  
> A controlled ablation (same model, two prompts) is the planned next step.


In [ ]:
import ollama

SUB_MULTIPLIER = {"low": 1.0, "medium": 0.5, "high": 0.1}


def classify_predictive(sentence: str, company_a: str, company_b: str) -> dict:
    """
    V3 classifier: reasons about *future demand transfer* rather than current state.

    Asks: if Company A grows, does Company B's demand structurally increase?
    Also captures substitutability — how easily A could replace B.

    Uses Llama 3.1 via local Ollama instance (no API key required).

    Returns:
        dict with keys: weight, demand_leverage, substitutability
    """
    prompt = f"""You are a financial analyst evaluating supply chain bottlenecks and hidden potential.
Context from Company A's 10-K filing: \"{sentence}\"

Based on this context, analyze the relationship from a forward-looking perspective:
If Company A ({company_a}) expands its business or the overall industry grows,
will the demand for Company B ({company_b})'s products/services inevitably increase?

Reply in JSON only with the following keys:
1. "demand_leverage_score": float 0.0-1.0
   - 1.0 = Highly critical. If A grows, A MUST buy more from B.
   - 0.5 = Moderate. B is helpful but growth is not strictly proportional.
   - 0.0 = Incidental mention. No structural demand link.
2. "substitutability": "low" | "medium" | "high"
   - "low" = B has monopoly or high switching costs. A cannot easily replace B.
   - "high" = B's product is a commodity. A can easily switch vendors.
3. "reasoning": brief economic logic (string)"""

    try:
        response = ollama.chat(
            model='llama3.1',
            messages=[{'role': 'user', 'content': prompt}],
            options={'format': 'json'}
        )
        result = json.loads(response['message']['content'])
        demand = float(result.get("demand_leverage_score", 0.0))
        sub = result.get("substitutability", "high")
        weight = demand * SUB_MULTIPLIER.get(sub, 0.1)
        return {"weight": weight, "demand_leverage": demand, "substitutability": sub}
    except Exception:
        return {"weight": 0.0, "demand_leverage": 0.0, "substitutability": "high"}


def run_predictive_llm_analysis(
    tickers: list,
    big_tech: list,
    snapshot_start: str = "2025-05-12",
    snapshot_end: str = "2025-05-17",
    filing_range: tuple = ("2024-01-01", "2025-05-15"),
    weight_threshold: float = 0.25
) -> pd.DataFrame:
    """
    Full pipeline using the predictive (V3) classifier.
    Identical structure to V2, with classify_predictive() replacing classify_structural().
    """
    total_capex = fetch_big_tech_capex(big_tech)
    if total_capex.empty:
        print("❌ Could not retrieve Big Tech CapEx data.")
        return pd.DataFrame()

    # ── Step 1: Build company profiles ───────────────────────────────────────
    print("Step 1: Building company profiles with historical P/E...")
    profiles = {}
    valid = [t for t in tickers if t in TICKER_TO_CIK]
    for t in valid:
        try:
            stock = yf.Ticker(t)
            hist = stock.history(start=snapshot_start, end=snapshot_end)
            price = hist['Close'].iloc[-1] if not hist.empty else None
            pe = 30
            fin = stock.financials.T
            fin.index = pd.to_datetime(fin.index).year
            if 2024 in fin.index and price:
                row = fin.loc[2024]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                eps = next((row[c] for c in fin.columns if 'EPS' in c or 'Earnings Per Share' in c), None)
                if (eps is None or pd.isna(eps) or eps <= 0) and 'Net Income' in row:
                    shares = row.get('Diluted Average Shares', row.get('Basic Average Shares', 1))
                    eps = row['Net Income'] / shares if shares > 0 else None
                if eps and eps > 0:
                    pe = price / eps
            profiles[t] = {'name': TICKER_TO_NAME[t], 'pe': pe, 'stock': stock}
        except Exception:
            continue
    print(f"  → {len(profiles)} companies profiled.")

    # ── Step 2: Extract relationships using predictive prompt ─────────────────
    print("Step 2: Scraping 10-Ks and running predictive inference...")
    edges = []
    for t, prof in profiles.items():
        url = get_10k_url(TICKER_TO_CIK[t], filing_range)
        if not url:
            continue
        try:
            time.sleep(0.2)
            res = requests.get(url, headers=HEADERS)
            if res.status_code != 200:
                continue
            text = re.sub(r'\s+', ' ', BeautifulSoup(res.text, 'html.parser').get_text(separator=' ').lower())
            for target, tprof in profiles.items():
                if target == t:
                    continue
                name = tprof['name'].lower()
                if name in text:
                    idx = text.find(name)
                    snippet = text[max(0, idx - 150): idx + 150]
                    result = classify_predictive(snippet, prof['name'], tprof['name'])
                    if result['weight'] > weight_threshold:
                        edges.append((t, target, result['weight']))
        except Exception:
            continue
    print(f"  → {len(edges)} edges extracted.")

    if not edges:
        print("❌ No edges found. Ensure Ollama is running: `ollama serve`")
        return pd.DataFrame()

    # ── Step 3: Build graph and compute scores ────────────────────────────────
    print("Step 3: Building graph and computing Bottleneck Scores...")
    G = nx.DiGraph()
    G.add_weighted_edges_from(edges)
    in_degrees = dict(G.in_degree(weight='weight'))
    max_in = max(in_degrees.values(), default=1)
    N = len(G.nodes())

    results = []
    for t in G.nodes():
        if t not in profiles:
            continue
        score = compute_bottleneck_score(t, profiles[t]['stock'], in_degrees.get(t, 0), max_in, profiles[t]['pe'], total_capex)
        if score is not None:
            results.append({
                'Ticker': t,
                'In_Degree_Weighted': round(in_degrees.get(t, 0), 3),
                'Potential_Centrality': round(in_degrees.get(t, 0) / (N - 1), 4) if N > 1 else 0,
                'PE': round(profiles[t]['pe'], 2),
                'Bottleneck_Score': score
            })

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results).set_index('Ticker')
    plot_network(G, df, profiles, "V3: Predictive LLM — Demand Leverage Network (Ollama Llama 3.1)")
    return df.sort_values('Bottleneck_Score', ascending=False)


# Run V3 analysis
print("Running V3: Predictive LLM (Ollama — local Llama 3.1)...")
v3_results = run_predictive_llm_analysis(IT_TICKERS, BIG_TECH)
if not v3_results.empty:
    print("\nTop 10 by Bottleneck Score (V3 — Predictive LLM):")
    print(v3_results.head(10))


## 4. Backtest: Keyword vs Structural LLM vs Predictive LLM

Compare the 1-year portfolio performance of the top 6 stocks selected by each approach,  
against the S&P 500 (SPY) as market benchmark.

**Update the portfolio lists below** with the actual top-6 tickers from your V2 and V3 outputs above.


In [ ]:
# ── Portfolio configuration ────────────────────────────────────────────────────
# Replace these with your actual top-6 outputs from V1 (keyword), V2, and V3.
KEYWORD_TOP6    = ['HPE', 'MSFT', 'PTC', 'AMAT', 'QCOM', 'AAPL']  # V1 keyword baseline
STRUCTURAL_TOP6 = ['HPE', 'MSFT', 'PTC', 'AMAT', 'QCOM', 'AAPL']  # V2 — update after running
PREDICTIVE_TOP6 = ['HPE', 'PTC', 'MSFT', 'AMAT', 'QCOM', 'AKAM']  # V3 — update after running

BACKTEST_START = "2025-05-18"
BACKTEST_END   = "2026-05-18"

all_tickers = list(set(KEYWORD_TOP6 + STRUCTURAL_TOP6 + PREDICTIVE_TOP6 + ['SPY']))

print(f"Downloading price data: {BACKTEST_START} → {BACKTEST_END}")
raw = yf.download(all_tickers, start=BACKTEST_START, end=BACKTEST_END, auto_adjust=True)
prices = raw['Close'].dropna()

# Normalize all prices to 100 at start (equal-weight simulation)
norm = prices / prices.iloc[0] * 100

kw_trend   = norm[KEYWORD_TOP6].mean(axis=1)
str_trend  = norm[STRUCTURAL_TOP6].mean(axis=1)
pred_trend = norm[PREDICTIVE_TOP6].mean(axis=1)
spy_trend  = norm['SPY']

kw_ret   = kw_trend.iloc[-1] - 100
str_ret  = str_trend.iloc[-1] - 100
pred_ret = pred_trend.iloc[-1] - 100
spy_ret  = spy_trend.iloc[-1] - 100

print("\n" + "="*55)
print("  1-Year Backtest Results (Equal-Weighted, May 2025–2026)")
print("="*55)
print(f"  V1 Keyword Centrality        : {kw_ret:+.2f}%")
print(f"  V2 Structural LLM (Groq)     : {str_ret:+.2f}%")
print(f"  V3 Predictive LLM (Ollama)   : {pred_ret:+.2f}%")
print(f"  Market Benchmark (SPY)        : {spy_ret:+.2f}%")
print("-"*55)
print(f"  Alpha (V3 vs SPY)             : {pred_ret - spy_ret:+.2f}%p")
print(f"  Alpha (V3 vs Keyword)         : {pred_ret - kw_ret:+.2f}%p")
print("="*55)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(kw_trend.index,   kw_trend   - 100, label=f'V1 Keyword [{kw_ret:+.2f}%]',   color='#1f77b4', linewidth=2)
ax.plot(str_trend.index,  str_trend  - 100, label=f'V2 Structural LLM [{str_ret:+.2f}%]', color='#ff7f0e', linewidth=2)
ax.plot(pred_trend.index, pred_trend - 100, label=f'V3 Predictive LLM [{pred_ret:+.2f}%]', color='#d62728', linewidth=3)
ax.plot(spy_trend.index,  spy_trend  - 100, label=f'SPY Benchmark [{spy_ret:+.2f}%]',   color='gray',    linewidth=1.5, linestyle='--')
ax.axhline(0, color='black', linewidth=1)
ax.set_title("Portfolio Cumulative Return — Keyword vs Structural LLM vs Predictive LLM (1 Year)", fontsize=14, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (%)")
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='upper left', fontsize=11, frameon=True)
plt.tight_layout()
plt.show()


## 5. Extended Backtest: 3-Year Annual Rebalancing (2020–2022)

Each year, the model is re-run on that year's 10-K filings to select a fresh top-6 portfolio,  
then held for 12 months before rebalancing again.

**Note:** This requires running `run_predictive_llm_analysis()` three times  
with different `filing_range` and `snapshot` parameters. Due to execution time,  
historical top-6 results are pre-filled below — replace them with your own outputs.


In [ ]:
# ── Pre-fill with your own top-6 outputs for each year ────────────────────────
# Run run_predictive_llm_analysis() with the parameters below for each year,
# then paste the resulting top-6 tickers into the dictionary.

# Parameters to use per year:
#   2020 filing: filing_range=("2020-01-01", "2020-06-01"), snapshot: 2020-04-25 to 2020-05-05
#   2021 filing: filing_range=("2021-01-01", "2021-06-01"), snapshot: 2021-04-25 to 2021-05-05
#   2022 filing: filing_range=("2022-01-01", "2022-06-01"), snapshot: 2022-04-25 to 2022-05-05

PORTFOLIO_BY_YEAR = {
    2020: ['MSFT', 'AAPL', 'NVDA', 'AMAT', 'QCOM', 'HPE'],  # Replace with your output
    2021: ['MSFT', 'AAPL', 'NVDA', 'AMAT', 'QCOM', 'HPE'],  # Replace with your output
    2022: ['MSFT', 'AAPL', 'NVDA', 'AMAT', 'QCOM', 'HPE'],  # Replace with your output
}

REBALANCE_PERIODS = [
    (2020, "2020-05-01", "2021-04-30"),
    (2021, "2021-05-01", "2022-04-30"),
    (2022, "2022-05-01", "2023-04-30"),
]

all_used = list(set(t for tickers in PORTFOLIO_BY_YEAR.values() for t in tickers) | {'SPY'})
print("Downloading 3-year price history (2020–2023)...")
price_data = yf.download(all_used, start="2020-05-01", end="2023-05-01", auto_adjust=True)['Close'].dropna()

# Chain daily returns across rebalancing periods
chained, spy_chained = [], []
for year, start, end in REBALANCE_PERIODS:
    period = price_data.loc[start:end]
    port_ret = period[PORTFOLIO_BY_YEAR[year]].pct_change().dropna().mean(axis=1)
    spy_ret  = period['SPY'].pct_change().dropna()
    chained.append(port_ret)
    spy_chained.append(spy_ret)

port_returns = pd.concat(chained)
spy_returns  = pd.concat(spy_chained)

# Cumulative wealth index (starts at 100)
port_cum = (1 + port_returns).cumprod() * 100
spy_cum  = (1 + spy_returns).cumprod() * 100

total_port = port_cum.iloc[-1] - 100
total_spy  = spy_cum.iloc[-1] - 100

print("\n" + "="*55)
print("  3-Year Rebalancing Backtest (2020–2022, Predictive LLM)")
print("="*55)
print(f"  LLM Potential Portfolio : {total_port:+.2f}%")
print(f"  SPY Benchmark           : {total_spy:+.2f}%")
print(f"  Alpha                   : {total_port - total_spy:+.2f}%p")
print("="*55)
print("⚠️  Note: 2020–2022 was predominantly a bull market for IT.")
print("   These results should be interpreted with caution.")

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(port_cum.index, port_cum - 100, label=f'LLM Potential Portfolio [{total_port:+.2f}%]', color='darkviolet', linewidth=3)
ax.plot(spy_cum.index,  spy_cum  - 100, label=f'SPY Benchmark [{total_spy:+.2f}%]',           color='gray',       linewidth=1.5, linestyle='--')
for _, rebal_date, _ in REBALANCE_PERIODS[1:]:
    ax.axvline(pd.to_datetime(rebal_date), color='red', linestyle=':', alpha=0.6, label='Rebalancing' if _ == REBALANCE_PERIODS[1][0] else '')
ax.axhline(0, color='black', linewidth=1)
ax.set_title("3-Year Multi-Period Rebalancing Backtest — Predictive LLM Portfolio (2020–2022)", fontsize=13, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (%)")
ax.grid(True, linestyle=':', alpha=0.5)
# Deduplicate legend entries
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()


## 6. [Optional] Individual Stock Price Validator

Utility cell to visualize the 1-year price trajectory for any list of tickers.  
Useful for manually verifying backtest results against actual stock performance.


In [ ]:
def plot_stock_history(tickers: list, period: str = '1y'):
    """
    Plot 1-year price charts for a list of tickers with annotated start/end prices.

    Args:
        tickers: List of ticker symbols.
        period: yfinance period string (default '1y').
    """
    for ticker in tickers:
        stock = yf.Ticker(ticker)
        df = stock.history(period=period).dropna(subset=['Close'])
        if len(df) < 2:
            print(f"[{ticker}] Insufficient data.")
            continue

        start_price = df['Close'].iloc[0]
        end_price   = df['Close'].iloc[-1]
        growth_rate = (end_price - start_price) / start_price * 100
        title_color = 'forestgreen' if growth_rate >= 0 else 'crimson'

        fig, ax = plt.subplots(figsize=(11, 5))
        ax.plot(df.index, df['Close'], color='royalblue', linewidth=2.5, label='Close Price')
        ax.scatter(df.index[0],  start_price, color='darkorange', s=100, zorder=5)
        ax.scatter(df.index[-1], end_price,   color='purple',     s=100, zorder=5)
        ax.annotate(f"${start_price:.2f}\n({df.index[0].strftime('%Y-%m-%d')})",
                    (df.index[0], start_price), xytext=(10, 0),
                    textcoords='offset points', ha='left', va='center',
                    fontsize=9, fontweight='bold', color='darkorange')
        ax.annotate(f"${end_price:.2f}\n({df.index[-1].strftime('%Y-%m-%d')})",
                    (df.index[-1], end_price), xytext=(-10, -15),
                    textcoords='offset points', ha='right', va='top',
                    fontsize=9, fontweight='bold', color='purple')
        ax.set_title(f"[{ticker}] 1-Year Price — Total Return: {growth_rate:+.2f}%",
                     fontsize=13, fontweight='bold', color=title_color)
        ax.set_xlabel("Date")
        ax.set_ylabel("Price ($)")
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.show()


# Validate the top candidates from each approach
plot_stock_history(['MSFT', 'PTC', 'HPE', 'AMAT', 'QCOM', 'AAPL'])
